# bilibili2txt Colab 服务端

此笔记本用于在 Google Colab 上启动远程 `b2t server run` 转写节点。所有的运行日志都会实时输出在下方单元格中。

In [ ]:
from google.colab import drive
import yaml
import subprocess
drive.mount('/content/drive')

try:
    with open('/content/drive/MyDrive/github/bilibili2txt/config.yaml', 'r', encoding='utf-8') as f:
        config_data = yaml.safe_load(f) or {}
    server_id = config_data.get('server', {}).get('server_id', 'digiplanets1')
except Exception:
    server_id = 'digiplanets1'
subprocess.run(['git', 'config', '--global', 'user.name', server_id])
subprocess.run(['git', 'config', '--global', 'user.email', f'{server_id}@protonmail.com'])

!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/digiplanets1_id_ed25519_github ~/.ssh/id_ed25519
!chmod 600 ~/.ssh/id_ed25519
!echo "Host github.com" > ~/.ssh/config
!echo "  HostName github.com" >> ~/.ssh/config
!echo "  User digiplanets1" >> ~/.ssh/config
!echo "  IdentityFile ~/.ssh/id_ed25519" >> ~/.ssh/config
!echo "  StrictHostKeyChecking accept-new" >> ~/.ssh/config
!export TZ=Asia/Shanghai
!chmod +x /content/drive/MyDrive/Faster-Whisper-XXL/faster-whisper-xxl

## 配置参数

请根据您的云盘路径和运行环境修改以下参数。

In [ ]:
REPO_URL = 'https://github.com/digiprospector/bilibili2txt.git'
PROJECT_DIR = '/content/drive/MyDrive/github/bilibili2txt'
CONFIG_PATH = f'{PROJECT_DIR}/config.yaml'
COLAB_PATH = '/content/drive/MyDrive/Colab Notebooks'

## 安装系统依赖

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg git

## 安装 Python 依赖库

In [ ]:
import os
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
!python -m pip install -U pip
!python -m pip install -r requirements.txt

## 环境与配置检查

In [ ]:
from pathlib import Path
import yaml

config_file = Path(CONFIG_PATH)
print('Config path:', config_file)
if not config_file.exists():
    raise FileNotFoundError(f'Config file not found: {config_file}')

config = yaml.safe_load(config_file.read_text(encoding='utf-8')) or {}
fw = Path(config.get('server', {}).get('faster_whisper_path', ''))
print('faster-whisper path:', fw)
if not fw.exists():
    raise FileNotFoundError(f'faster-whisper path not found: {fw}')

queue_dir = Path(config.get('queue', {}).get('repo_dir', 'queue'))
data_dir = Path(config.get('data', {}).get('repo_dir', 'data'))
print('Queue repo dir:', queue_dir)
print('Data repo dir:', data_dir)
print('如果这些路径是相对路径，b2t 运行时会相对于项目根目录进行解析。')


## 启动转写服务端

In [ ]:
!python b2t.py --config {CONFIG_PATH} server run